# Download and process TESSERA embeddings for Bavaria

This notebook handles the full TESSERA workflow for Bavaria:

1. prepare the Bavaria area of interest,
2. download annual embeddings for 2017–2024,
3. check whether any NPY tiles are missing and retry them,
4. convert the NPY files to georeferenced GeoTIFFs,
5. build annual Bavaria mosaics in EPSG:3035,
6. inspect the embeddings through time using a shared PCA,
7. map changes between consecutive years.

The workflow is split into sections so that the download, conversion, and visualization steps can also be run separately. This is useful because none of them is exactly a five-minute job.

## 1. Environment and imports

Install packages only when they are missing. A kernel restart may be needed afterwards.

In [1]:
# Uncomment only if the packages are missing:
# %pip install --upgrade geotessera geopandas pyogrio rasterio tqdm matplotlib scikit-learn imageio

from pathlib import Path
from collections import defaultdict
import gc
import json
import math
import re
import shutil
import subprocess
import sys
import time

import geopandas as gpd
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.enums import Resampling
from sklearn.decomposition import PCA
from tqdm.auto import tqdm

from geotessera import GeoTessera
from geotessera.tiles import discover_npy_tiles

print("Python:", sys.version)
print("GeoTessera CLI:", shutil.which("geotessera"))


Python: 3.11.15 | packaged by conda-forge | (main, Mar  5 2026, 16:45:40) [GCC 14.3.0]
GeoTessera CLI: /mnt/dss_project/lmandl/anaconda3/envs/terrabyte/bin/geotessera


## 2. Configuration

Set the AOI file, output location, years, and download options here. The default setup processes 2017–2024.

In [2]:
AOI_FILE = Path("/mnt/dss_project/lmandl/_terrabyte/data_bavaria/bavaria.gpkg")
AOI_LAYER = None  # Only set a layer name if the GeoPackage contains more than one.

OUTPUT_ROOT = Path("/mnt/dss_project/lmandl/TESSERA_bavaria")

YEARS = list(range(2017, 2024))

# NPY is required for the later tile validation and conversion workflow.
DOWNLOAD_FORMAT = "npy"
BANDS = None

PAUSE_BETWEEN_YEARS_SECONDS = 2

RUN_TEST_ONLY = False
TEST_YEAR = 2024

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("AOI:", AOI_FILE)
print("Output root:", OUTPUT_ROOT)
print("Years:", YEARS)
print("Download format:", DOWNLOAD_FORMAT)


AOI: /mnt/dss_project/lmandl/_terrabyte/data_bavaria/bavaria.gpkg
Output root: /mnt/dss_project/lmandl/TESSERA_bavaria
Years: [2017, 2018, 2019, 2020, 2021, 2022, 2023]
Download format: npy


## 3. Prepare the Bavaria AOI

The AOI is checked, transformed to WGS84, dissolved into one geometry, and written to GeoJSON for the GeoTessera command-line interface.

In [3]:
if not AOI_FILE.exists():
    raise FileNotFoundError(
        f"AOI not found: {AOI_FILE}\n"
        "Update AOI_FILE before continuing."
    )

read_kwargs = {}
if AOI_LAYER is not None:
    read_kwargs["layer"] = AOI_LAYER

aoi = gpd.read_file(AOI_FILE, **read_kwargs)

if aoi.empty:
    raise ValueError("The AOI file does not contain any geometries.")

aoi = aoi[aoi.geometry.notna()].copy()
aoi = aoi[~aoi.geometry.is_empty].copy()

if aoi.crs is None:
    raise ValueError(
        "The AOI has no CRS information. "
        "Assign the correct CRS before exporting it."
    )

try:
    aoi["geometry"] = aoi.geometry.make_valid()
except AttributeError:
    aoi["geometry"] = aoi.buffer(0)

aoi_wgs84 = aoi.to_crs(4326)

try:
    merged_geometry = aoi_wgs84.geometry.union_all()
except AttributeError:
    merged_geometry = aoi_wgs84.unary_union

aoi_merged = gpd.GeoDataFrame(
    {"name": ["Bavaria"]},
    geometry=[merged_geometry],
    crs="EPSG:4326",
)

PREPARED_AOI = OUTPUT_ROOT / "_aoi" / "bavaria_wgs84.geojson"
PREPARED_AOI.parent.mkdir(parents=True, exist_ok=True)
aoi_merged.to_file(PREPARED_AOI, driver="GeoJSON")

bounds = aoi_merged.total_bounds
AOI_BOUNDS = tuple(float(value) for value in bounds)

print("Prepared AOI:", PREPARED_AOI)
print(
    "WGS84 bounds:",
    f"min_lon={bounds[0]:.4f}, min_lat={bounds[1]:.4f}, "
    f"max_lon={bounds[2]:.4f}, max_lat={bounds[3]:.4f}",
)

aoi_merged


Prepared AOI: /mnt/dss_project/lmandl/TESSERA_bavaria/_aoi/bavaria_wgs84.geojson
WGS84 bounds: min_lon=8.9764, min_lat=47.2701, max_lon=13.8396, max_lat=50.5647


,name,geometry
0,Bavaria,"POLYGON ((9.09498 49.58647, 9.09457 49.58668, ..."


## 4. Check GeoTessera and inspect coverage

First, make sure the command-line tool is available. The optional coverage map is a quick visual check before starting several hundred gigabytes of downloads.

In [4]:
def run_command(command, check=True, capture_output=False):
    command = [str(item) for item in command]
    print("\n$", " ".join(command), flush=True)

    return subprocess.run(
        command,
        check=check,
        text=True,
        capture_output=capture_output,
    )


if shutil.which("geotessera") is None:
    raise RuntimeError(
        "The GeoTessera CLI was not found in the active environment. "
        "Restart the kernel after installation and run the notebook again."
    )

run_command(["geotessera", "--help"])



$ geotessera --help
                                                                                
 Usage: geotessera [OPTIONS] COMMAND [ARGS]...                                  
                                                                                
 GeoTessera v0.7.5: Download satellite embedding tiles as GeoTIFFs              
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --help          Show this message and exit.                                  │
╰──────────────────────────────────────────────────────────────────────────────╯
╭─ Commands ───────────────────────────────────────────────────────────────────╮
│ info       Show information about tile files or library.                     │
│ coverage   Generate coverage visualizations showing Tessera embedding        │
│            availability.                                                     │
│ downl

CompletedProcess(args=['geotessera', '--help'], returncode=0)

In [5]:
COVERAGE_DIR = OUTPUT_ROOT / "_coverage"
COVERAGE_DIR.mkdir(parents=True, exist_ok=True)

coverage_png = COVERAGE_DIR / "bavaria_tessera_coverage.png"

coverage_result = run_command(
    [
        "geotessera",
        "coverage",
        "--region-file", PREPARED_AOI,
        "--output", coverage_png,
    ],
    check=False,
)

if coverage_result.returncode != 0:
    print(
        "Coverage-map creation failed. "
        "This is annoying, but it does not prevent the actual download."
    )
else:
    print("Coverage map:", coverage_png)



$ geotessera coverage --region-file /mnt/dss_project/lmandl/TESSERA_bavaria/_aoi/bavaria_wgs84.geojson --output /mnt/dss_project/lmandl/TESSERA_bavaria/_coverage/bavaria_tessera_coverage.png
Region bounding box: [8.976365°E, 47.270113°N] - [13.839635°E, 50.564699°N]
INFO     Using cached registry: /home/lmandl/.cache/geotessera/registry.parquet 
INFO     Checking for registry updates...                                       
INFO     Verified with server - registry is current (no download needed)        
INFO     Loaded GeoParquet with 4,397,409 tiles                                 
INFO     Using cached landmasks registry:                                       
         /home/lmandl/.cache/geotessera/landmasks.parquet                       
INFO     Checking for landmasks registry updates...                             
INFO     Verified with server - landmasks registry is current (no download      
         needed)                                                                
🔄 G

## 5. Download annual embeddings

Each year is written to its own directory. Progress is logged after every year, so an interrupted run does not leave us guessing what actually finished.

In [ ]:
def build_download_command(year):
    year_dir = OUTPUT_ROOT / str(year)
    year_dir.mkdir(parents=True, exist_ok=True)

    command = [
        "geotessera",
        "download",
        "--region-file", PREPARED_AOI,
        "--year", str(year),
        "--format", DOWNLOAD_FORMAT,
        "--output", year_dir,
    ]

    if BANDS is not None:
        if DOWNLOAD_FORMAT != "tiff":
            raise ValueError(
                "Band selection is only supported here for TIFF downloads. "
                "Use BANDS=None for the NPY workflow."
            )
        command.extend(["--bands", ",".join(map(str, BANDS))])

    return command, year_dir


def download_year(year):
    command, year_dir = build_download_command(year)
    started = time.time()

    result = run_command(command, check=False)
    elapsed_minutes = (time.time() - started) / 60

    return {
        "year": int(year),
        "success": result.returncode == 0,
        "returncode": int(result.returncode),
        "output_directory": str(year_dir),
        "elapsed_minutes": round(elapsed_minutes, 2),
    }


In [ ]:
if RUN_TEST_ONLY:
    print(json.dumps(download_year(TEST_YEAR), indent=2))
else:
    log_dir = OUTPUT_ROOT / "_logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    status_file = log_dir / "download_status.json"

    download_results = []

    for index, year in enumerate(YEARS, start=1):
        print("\n" + "=" * 80)
        print(f"Year {year} ({index}/{len(YEARS)})")
        print("=" * 80)

        result = download_year(year)
        download_results.append(result)

        status_file.write_text(
            json.dumps(download_results, indent=2),
            encoding="utf-8",
        )

        if result["success"]:
            print(f"Year {year} completed successfully.")
        else:
            print(
                f"Year {year} failed with return code "
                f"{result['returncode']}."
            )

        if index < len(YEARS):
            time.sleep(PAUSE_BETWEEN_YEARS_SECONDS)

    print("\nDownload status:")
    print(json.dumps(download_results, indent=2))
    print("Status log:", status_file)


## 6. Check and retry missing NPY tiles

This step compares the expected registry tiles with the files on disk. It checks file names and coordinates only, so the 128-dimensional arrays stay safely out of memory for now.

In [ ]:
gt = GeoTessera()


def find_missing_tiles(year, bounds=AOI_BOUNDS):
    expected = gt.registry.load_blocks_for_region(
        bounds=bounds,
        year=year,
    )

    npy_root = (
        OUTPUT_ROOT
        / str(year)
        / "global_0.1_degree_representation"
        / str(year)
    )

    existing = set()

    if npy_root.exists():
        for path in npy_root.rglob("grid_*.npy"):
            if path.name.endswith("_scales.npy"):
                continue

            match = re.match(
                r"grid_(-?\d+(?:\.\d+)?)_(-?\d+(?:\.\d+)?)\.npy$",
                path.name,
            )

            if match:
                existing.add(
                    (
                        round(float(match.group(1)), 2),
                        round(float(match.group(2)), 2),
                    )
                )

    expected_coords = {
        (
            round(float(row[1]), 2),
            round(float(row[2]), 2),
        )
        for row in expected
    }

    missing = sorted(expected_coords - existing)

    return {
        "year": int(year),
        "expected": len(expected_coords),
        "existing": len(existing),
        "missing": missing,
        "npy_root": str(npy_root),
    }


missing_tile_summary = []

for year in YEARS:
    result = find_missing_tiles(year)
    missing_tile_summary.append(result)

    print(
        f"{year}: expected={result['expected']}, "
        f"existing={result['existing']}, "
        f"missing={len(result['missing'])}"
    )

    if result["missing"]:
        print("  First missing tiles:", result["missing"][:10])


In [ ]:
def retry_missing_tiles(year, missing_tiles, pause_seconds=0.5):
    output_dir = OUTPUT_ROOT / str(year)
    output_dir.mkdir(parents=True, exist_ok=True)

    failed_log = output_dir / "failed_tiles.json"
    failed = []

    for index, (lon, lat) in enumerate(missing_tiles, start=1):
        print(
            f"\n{year} | tile {index}/{len(missing_tiles)}: "
            f"lon={lon}, lat={lat}"
        )

        command = [
            "geotessera",
            "download",
            "--tile", f"{lon},{lat}",
            "--year", str(year),
            "--format", "npy",
            "--output", str(output_dir),
        ]

        result = subprocess.run(command)

        if result.returncode != 0:
            failed.append(
                {
                    "lon": lon,
                    "lat": lat,
                    "returncode": result.returncode,
                }
            )

            failed_log.write_text(
                json.dumps(failed, indent=2),
                encoding="utf-8",
            )

        time.sleep(pause_seconds)

    if not failed:
        failed_log.unlink(missing_ok=True)

    return failed


retry_summary = []

for item in missing_tile_summary:
    year = item["year"]
    missing_tiles = item["missing"]

    if not missing_tiles:
        retry_summary.append(
            {"year": year, "retried": 0, "failed": 0}
        )
        continue

    print("\n" + "=" * 80)
    print(f"Retrying {len(missing_tiles)} missing tiles for {year}")
    print("=" * 80)

    failed = retry_missing_tiles(year, missing_tiles)

    retry_summary.append(
        {
            "year": year,
            "retried": len(missing_tiles),
            "failed": len(failed),
        }
    )

retry_log = OUTPUT_ROOT / "_logs" / "missing_tile_retry_status.json"
retry_log.parent.mkdir(parents=True, exist_ok=True)
retry_log.write_text(
    json.dumps(retry_summary, indent=2),
    encoding="utf-8",
)

print("\nRetry summary:")
print(json.dumps(retry_summary, indent=2))
print("Retry log:", retry_log)


## 7. Summarize the local download

Count the downloaded files and report the storage used for each year.

In [ ]:
def directory_size_bytes(path):
    total = 0

    for file_path in path.rglob("*"):
        if file_path.is_file():
            try:
                total += file_path.stat().st_size
            except FileNotFoundError:
                pass

    return total


download_summary = []

for year in YEARS:
    year_dir = OUTPUT_ROOT / str(year)
    files = (
        [path for path in year_dir.rglob("*") if path.is_file()]
        if year_dir.exists()
        else []
    )

    size_bytes = (
        directory_size_bytes(year_dir)
        if year_dir.exists()
        else 0
    )

    download_summary.append(
        {
            "year": year,
            "files": len(files),
            "size_gb": round(size_bytes / 1024**3, 2),
            "directory": str(year_dir),
        }
    )

download_summary


# Convert NPY embeddings to annual Bavaria mosaics

The next part dequantizes the NPY embeddings, writes georeferenced tile GeoTIFFs, creates source VRTs by native CRS, and warps everything into one Bavaria mosaic per year in EPSG:3035.

Bavaria crosses more than one UTM zone, so the source rasters are kept separate by CRS until the final warp. Pretending otherwise would make life simpler, but not the data better.

    The de-quantization take some time!

## 8. Conversion and mosaic settings

Define the input folders, output folders, target CRS, resolution, and processing years.

In [ ]:
# Folder containing the annual NPY downloads.
NPY_ROOT = OUTPUT_ROOT

# Keep intermediate tiles and final mosaics in separate folders.
TILE_TIFF_ROOT = OUTPUT_ROOT / "_tile_geotiffs"
MOSAIC_ROOT = OUTPUT_ROOT / "mosaics_epsg3035"
VRT_ROOT = OUTPUT_ROOT / "_vrt"

TILE_TIFF_ROOT.mkdir(parents=True, exist_ok=True)
MOSAIC_ROOT.mkdir(parents=True, exist_ok=True)
VRT_ROOT.mkdir(parents=True, exist_ok=True)

# Common target CRS for the Bavaria mosaic.
TARGET_CRS = "EPSG:3035"
TARGET_RESOLUTION = 10

# Keep all 128 representation dimensions.
MOSAIC_BANDS = None

# Compression for temporary tile GeoTIFFs.
# ZSTD requires a reasonably recent GDAL version.
TILE_COMPRESSION = "ZSTD"
TILE_ZSTD_LEVEL = 9

# Compression for the final mosaic.
MOSAIC_COMPRESSION = "ZSTD"
MOSAIC_ZSTD_LEVEL = 9

# Delete temporary tile GeoTIFFs after a successful annual mosaic.
# Keep this False when the individual tiles may be useful later.
DELETE_TILE_TIFFS_AFTER_MOSAIC = False

# Skip existing outputs when they pass basic validation.
SKIP_EXISTING_TILE_TIFFS = True
SKIP_EXISTING_MOSAICS = True

# GDAL working memory in MB.
# Avoid heroic values when other processes share the machine.
GDAL_CACHE_MB = 4096

# Optionally test the workflow on one year.
MOSAIC_TEST_ONLY = False
MOSAIC_TEST_YEAR = 2024

# Years to process:
MOSAIC_YEARS = [MOSAIC_TEST_YEAR] if MOSAIC_TEST_ONLY else YEARS

print("Years:", MOSAIC_YEARS)
print("Target CRS:", TARGET_CRS)
print("Target resolution:", TARGET_RESOLUTION, "m")
print("Final output directory:", MOSAIC_ROOT)


In [ ]:
required_programs = ["gdalbuildvrt", "gdalwarp"]
missing_programs = [
    program
    for program in required_programs
    if shutil.which(program) is None
]

if missing_programs:
    raise RuntimeError(
        "The following GDAL programs were not found: "
        + ", ".join(missing_programs)
        + ". Install GDAL in the active Conda environment, for example with "
          "`conda install -c conda-forge gdal`."
    )

for program in required_programs:
    print(f"{program}: {shutil.which(program)}")


## 9. Inspect one NPY year

Read one example tile with memory mapping to check its shape, data type, metadata, and native projection before converting the full archive.

In [ ]:
def inspect_npy_year(year):
    year_dir = NPY_ROOT / str(year)

    if not year_dir.exists():
        raise FileNotFoundError(f"Year directory not found: {year_dir}")

    tiles = discover_npy_tiles(year_dir)

    if not tiles:
        raise RuntimeError(
            f"No complete NPY tiles für {year} in {year_dir} gefunden. "
            "To be expected: Embedding-, scale-, and land mask files."
        )

    crs_counts = defaultdict(int)
    for tile in tiles:
        crs_counts[str(tile.crs)] += 1

    first = tiles[0]
    quantized = np.load(first._embedding_path, mmap_mode="r")
    scales = np.load(first._scales_path, mmap_mode="r")

    print(f"Year: {year}")
    print(f"Tiles found: {len(tiles)}")
    print("Native projections:")
    for crs, count in sorted(crs_counts.items()):
        print(f"  {crs}: {count} tiles")
    print("Example tile:", first.grid_name)
    print("Quantized data:", quantized.shape, quantized.dtype)
    print("Scale array:", scales.shape, scales.dtype)
    print("Raster size:", first.height, first.width)
    print("Transform:", first.transform)

    return tiles


inspection_year = MOSAIC_YEARS[0]
inspection_tiles = inspect_npy_year(inspection_year)


## 10. Convert NPY tiles to georeferenced GeoTIFFs

Each NPY tile is dequantized and written as a multiband GeoTIFF in its native projection.

In [ ]:
def tile_output_path(tile, year):
    crs_text = tile.crs.to_string() if hasattr(tile.crs, "to_string") else str(tile.crs)
    epsg_match = re.search(r"(\d+)$", crs_text)
    crs_folder = f"EPSG_{epsg_match.group(1)}" if epsg_match else crs_text.replace(":", "_")

    return (
        TILE_TIFF_ROOT
        / str(year)
        / crs_folder
        / f"{tile.grid_name}_{year}_tessera.tif"
    )


def valid_existing_tiff(path, expected_count):
    if not path.exists():
        return False

    try:
        with rasterio.open(path) as src:
            return (
                src.count == expected_count
                and src.width > 0
                and src.height > 0
                and src.crs is not None
            )
    except Exception:
        return False


def write_tile_geotiff(tile, output_path, bands=None):
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # GeoTessera dequantizes internally:
    # float32_embedding = int8_embedding * scale
    embedding = tile.load_embedding()

    if embedding.ndim != 3:
        raise ValueError(
            f"Unexpected embedding shape für {tile.grid_name}: {embedding.shape}"
        )

    if bands is None:
        selected = embedding
        band_indices = list(range(embedding.shape[2]))
    else:
        band_indices = list(bands)
        selected = embedding[:, :, band_indices]

    band_count = selected.shape[2]

    profile = {
        "driver": "GTiff",
        "height": tile.height,
        "width": tile.width,
        "count": band_count,
        "dtype": "float32",
        "crs": tile.crs,
        "transform": tile.transform,
        "tiled": True,
        "blockxsize": 256,
        "blockysize": 256,
        "compress": TILE_COMPRESSION,
        "zstd_level": TILE_ZSTD_LEVEL,
        "predictor": 3,
        "BIGTIFF": "YES",
        "interleave": "band",
    }

    temporary_path = output_path.with_suffix(".part.tif")
    temporary_path.unlink(missing_ok=True)

    try:
        with rasterio.Env(GDAL_TIFF_INTERNAL_MASK=True):
            with rasterio.open(temporary_path, "w", **profile) as dst:
                # Rasterio expects (band, row, column).
                dst.write(np.moveaxis(selected, 2, 0))

                for output_band, original_band in enumerate(band_indices, start=1):
                    dst.set_band_description(
                        output_band,
                        f"Tessera_representation_{original_band:03d}",
                    )

                dst.update_tags(
                    TESSERA_YEAR=str(tile.year),
                    TESSERA_TILE_LON=f"{tile.lon:.2f}",
                    TESSERA_TILE_LAT=f"{tile.lat:.2f}",
                    TESSERA_SOURCE_FORMAT="quantized NPY plus scale",
                    TESSERA_DEQUANTIZATION="float32 = int8 * scale",
                    TESSERA_REPRESENTATION_COUNT=str(band_count),
                )

                # Store the land mask as an internal GDAL mask band.
                with rasterio.open(tile._landmask_path) as landmask_src:
                    landmask = landmask_src.read(1)
                    valid_mask = np.where(landmask > 0, 255, 0).astype("uint8")
                    dst.write_mask(valid_mask)

        temporary_path.replace(output_path)

    except Exception:
        temporary_path.unlink(missing_ok=True)
        raise

    finally:
        # Release large arrays immediately. RAM also deserves boundaries.
        del embedding
        if "selected" in locals():
            del selected

    return output_path


def convert_year_tiles_to_tiffs(year):
    year_dir = NPY_ROOT / str(year)
    tiles = discover_npy_tiles(year_dir)

    if not tiles:
        raise RuntimeError(f"No NPY tiles für {year} found: {year_dir}")

    expected_count = 128 if MOSAIC_BANDS is None else len(MOSAIC_BANDS)
    created_or_found = []

    for tile in tqdm(tiles, desc=f"NPY → GeoTIFF {year}"):
        output_path = tile_output_path(tile, year)

        if (
            SKIP_EXISTING_TILE_TIFFS
            and valid_existing_tiff(output_path, expected_count)
        ):
            created_or_found.append(output_path)
            continue

        created_or_found.append(
            write_tile_geotiff(
                tile=tile,
                output_path=output_path,
                bands=MOSAIC_BANDS,
            )
        )

    return created_or_found


## 11. Create source VRTs by native projection

Bavaria intersects several native projections. Building one VRT per CRS keeps that explicit and avoids mixing grids too early.

In [ ]:
def get_raster_crs_key(tiff_path):
    with rasterio.open(tiff_path) as src:
        if src.crs is None:
            raise ValueError(f"GeoTIFF without CRS: {tiff_path}")
        return src.crs.to_string()


def create_source_vrts(year, tile_tiffs):
    grouped = defaultdict(list)

    for tiff_path in tile_tiffs:
        grouped[get_raster_crs_key(tiff_path)].append(Path(tiff_path))

    year_vrt_dir = VRT_ROOT / str(year)
    year_vrt_dir.mkdir(parents=True, exist_ok=True)

    vrt_paths = []

    for crs_text, paths in sorted(grouped.items()):
        safe_crs = crs_text.replace(":", "_").replace("/", "_")
        file_list = year_vrt_dir / f"files_{safe_crs}.txt"
        vrt_path = year_vrt_dir / f"tessera_{year}_{safe_crs}.vrt"

        file_list.write_text(
            "\n".join(str(path.resolve()) for path in paths) + "\n",
            encoding="utf-8",
        )

        command = [
            "gdalbuildvrt",
            "-overwrite",
            "-resolution", "highest",
            "-input_file_list", file_list,
            vrt_path,
        ]

        result = run_command(command, check=False)

        if result.returncode != 0 or not vrt_path.exists():
            raise RuntimeError(
                f"VRT creation für {year}, {crs_text} failed."
            )

        vrt_paths.append(vrt_path)

    print(f"{year}: created {len(vrt_paths)} source VRT(s).")
    return vrt_paths


## 12. Build annual EPSG:3035 mosaics

Warp the source VRTs to the common European projection, clip them to Bavaria, and write one mosaic per year.

In [ ]:
def valid_existing_mosaic(path, expected_count):
    if not path.exists():
        return False

    try:
        with rasterio.open(path) as src:
            return (
                src.count == expected_count
                and src.crs is not None
                and src.crs.to_string().upper() == TARGET_CRS.upper()
                and src.width > 0
                and src.height > 0
            )
    except Exception:
        return False


def build_bavaria_mosaic(year, source_vrts):
    band_count = 128 if MOSAIC_BANDS is None else len(MOSAIC_BANDS)

    output_path = (
        MOSAIC_ROOT
        / f"tessera_bavaria_{year}_{band_count}bands_epsg3035_10m.tif"
    )

    if (
        SKIP_EXISTING_MOSAICS
        and valid_existing_mosaic(output_path, band_count)
    ):
        print(f"{year}: skipping existing mosaic: {output_path}")
        return output_path

    temporary_path = output_path.with_suffix(".part.tif")
    temporary_path.unlink(missing_ok=True)

    command = [
        "gdalwarp",
        "-overwrite",
        "-of", "GTiff",
        "-t_srs", TARGET_CRS,
        "-tr", str(TARGET_RESOLUTION), str(TARGET_RESOLUTION),
        "-tap",
        "-r", "bilinear",
        "-cutline", PREPARED_AOI,
        "-crop_to_cutline",
        "-multi",
        "-wo", "NUM_THREADS=ALL_CPUS",
        "-wm", str(GDAL_CACHE_MB),
        "-co", "TILED=YES",
        "-co", "BLOCKXSIZE=256",
        "-co", "BLOCKYSIZE=256",
        "-co", f"COMPRESS={MOSAIC_COMPRESSION}",
        "-co", f"ZSTD_LEVEL={MOSAIC_ZSTD_LEVEL}",
        "-co", "PREDICTOR=3",
        "-co", "BIGTIFF=YES",
        "-co", "NUM_THREADS=ALL_CPUS",
        "-co", "SPARSE_OK=TRUE",
    ]

    command.extend(source_vrts)
    command.append(temporary_path)

    result = run_command(command, check=False)

    if result.returncode != 0 or not temporary_path.exists():
        temporary_path.unlink(missing_ok=True)
        raise RuntimeError(f"Mosaic creation für {year} failed.")

    # Add band descriptions and metadata.
    with rasterio.open(temporary_path, "r+") as dst:
        original_bands = (
            list(range(128))
            if MOSAIC_BANDS is None
            else list(MOSAIC_BANDS)
        )

        for output_band, original_band in enumerate(original_bands, start=1):
            dst.set_band_description(
                output_band,
                f"Tessera_representation_{original_band:03d}",
            )

        dst.update_tags(
            TESSERA_YEAR=str(year),
            TESSERA_REGION="Bavaria",
            TESSERA_TARGET_CRS=TARGET_CRS,
            TESSERA_RESOLUTION_METRES=str(TARGET_RESOLUTION),
            TESSERA_REPRESENTATION_COUNT=str(band_count),
            TESSERA_RESAMPLING="bilinear",
            TESSERA_SOURCE_FORMAT="dequantized float32 GeoTIFF tiles",
        )

    temporary_path.replace(output_path)

    print(f"{year}: mosaic created: {output_path}")
    return output_path


## 13. Process all years

Run the conversion and mosaic workflow for every selected year. Existing valid outputs are reused where possible, because recomputing everything for sport would be a bit much.

In [ ]:
import gc

mosaic_results = []

for year in MOSAIC_YEARS:
    print("\n" + "=" * 90)
    print(f"TESSERA conversion and Bavaria mosaic: {year}")
    print("=" * 90)

    final_mosaic = (
        MOSAIC_ROOT
        / (
            f"tessera_bavaria_{year}_"
            f"{128 if MOSAIC_BANDS is None else len(MOSAIC_BANDS)}bands_"
            "epsg3035_10m.tif"
        )
    )

    # If the final product already exists and is valid,
    # the expensive tile conversion can be skipped.
    if (
        SKIP_EXISTING_MOSAICS
        and valid_existing_mosaic(
            final_mosaic,
            128 if MOSAIC_BANDS is None else len(MOSAIC_BANDS),
        )
    ):
        print(f"{year}: finished mosaic already exists.")
        mosaic_results.append({
            "year": year,
            "success": True,
            "mosaic": str(final_mosaic),
            "status": "already_exists",
        })
        continue

    try:
        tile_tiffs = convert_year_tiles_to_tiffs(year)
        source_vrts = create_source_vrts(year, tile_tiffs)
        mosaic_path = build_bavaria_mosaic(year, source_vrts)

        mosaic_results.append({
            "year": year,
            "success": True,
            "mosaic": str(mosaic_path),
            "tile_count": len(tile_tiffs),
            "status": "created",
        })

        if DELETE_TILE_TIFFS_AFTER_MOSAIC:
            year_tile_dir = TILE_TIFF_ROOT / str(year)
            if year_tile_dir.exists():
                shutil.rmtree(year_tile_dir)
                print(f"{year}: temporary tile GeoTIFFs deleted.")

    except Exception as error:
        mosaic_results.append({
            "year": year,
            "success": False,
            "error": repr(error),
        })
        print(f"{year}: ERROR: {error}")

    finally:
        gc.collect()

    log_file = OUTPUT_ROOT / "_logs" / "mosaic_status.json"
    log_file.parent.mkdir(parents=True, exist_ok=True)
    log_file.write_text(
        json.dumps(mosaic_results, indent=2),
        encoding="utf-8",
    )

print("\nProcessing completed.")
print(json.dumps(mosaic_results, indent=2))


## 14. Validate the annual mosaics

Check that every expected mosaic exists and report its dimensions, band count, data type, CRS, and file size.

In [ ]:
mosaic_summary = []

for year in MOSAIC_YEARS:
    candidates = sorted(
        MOSAIC_ROOT.glob(f"tessera_bavaria_{year}_*bands_epsg3035_10m.tif")
    )

    if not candidates:
        mosaic_summary.append({
            "year": year,
            "exists": False,
        })
        continue

    mosaic_path = candidates[0]

    with rasterio.open(mosaic_path) as src:
        mosaic_summary.append({
            "year": year,
            "exists": True,
            "path": str(mosaic_path),
            "size_gb": round(mosaic_path.stat().st_size / 1024**3, 2),
            "bands": src.count,
            "dtype": src.dtypes[0],
            "crs": src.crs.to_string() if src.crs else None,
            "resolution": src.res,
            "width": src.width,
            "height": src.height,
            "bounds": tuple(round(value, 2) for value in src.bounds),
            "compression": src.compression.value if src.compression else None,
        })

mosaic_summary


## Runtime benchmark for one year

This section is separate from the regular multi-year workflow. It is meant for measuring the runtime of one representative year after the full dataset has already been downloaded.

Choose one year below and use separate benchmark folders. This avoids overwriting the existing archive and, just as importantly, prevents the script from simply skipping files that are already there.

In [6]:
# Year used for the runtime benchmark.
RUNTIME_YEAR = 2022

# Separate folders for the benchmark run.
# Existing production data remain untouched.
RUNTIME_DOWNLOAD_ROOT = OUTPUT_ROOT / "_runtime_test" / "downloads"
RUNTIME_TILE_TIFF_ROOT = OUTPUT_ROOT / "_runtime_test" / "tile_geotiffs"

RUNTIME_DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)
RUNTIME_TILE_TIFF_ROOT.mkdir(parents=True, exist_ok=True)

print("Runtime test year:", RUNTIME_YEAR)
print("Download test folder:", RUNTIME_DOWNLOAD_ROOT)
print("GeoTIFF test folder:", RUNTIME_TILE_TIFF_ROOT)

Runtime test year: 2022
Download test folder: /mnt/dss_project/lmandl/TESSERA_bavaria/_runtime_test/downloads
GeoTIFF test folder: /mnt/dss_project/lmandl/TESSERA_bavaria/_runtime_test/tile_geotiffs


### Measure the download time

This repeats the download for one year in a separate folder. The result is the actual GeoTessera download time for that year, not just the time needed to notice that the files already exist.

In [ ]:
import time
import subprocess

runtime_year_dir = RUNTIME_DOWNLOAD_ROOT / str(RUNTIME_YEAR)
runtime_year_dir.mkdir(parents=True, exist_ok=True)

runtime_download_command = [
    "geotessera",
    "download",
    "--region-file", PREPARED_AOI,
    "--year", str(RUNTIME_YEAR),
    "--format", DOWNLOAD_FORMAT,
    "--output", str(runtime_year_dir),
]

if BANDS is not None:
    if DOWNLOAD_FORMAT != "tiff":
        raise ValueError(
            "Band selection is only supported for TIFF downloads. "
            "Use BANDS=None for the NPY workflow."
        )
    runtime_download_command.extend(
        ["--bands", ",".join(map(str, BANDS))]
    )

download_start = time.perf_counter()

download_result = subprocess.run(
    runtime_download_command,
    check=False,
)

download_seconds = time.perf_counter() - download_start
download_minutes = download_seconds / 60
download_hours = download_seconds / 3600

print()
print("Download return code:", download_result.returncode)
print(
    f"TESSERA download for {RUNTIME_YEAR} took "
    f"{download_hours:.2f} hours "
    f"({download_minutes:.1f} minutes)."
)

INFO     Using cached registry: /home/lmandl/.cache/geotessera/registry.parquet 
INFO     Checking for registry updates...                                       
INFO     Verified with server - registry is current (no download needed)        
INFO     Loaded GeoParquet with 4,397,409 tiles                                 
INFO     Using cached landmasks registry:                                       
         /home/lmandl/.cache/geotessera/landmasks.parquet                       
INFO     Checking for landmasks registry updates...                             
INFO     Verified with server - landmasks registry is current (no download      
         needed)                                                                
Calculated bbox from 
/mnt/dss_project/lmandl/TESSERA_bavaria/_aoi/bavaria_wgs84.geojson: [8.976365°E,
47.270113°N] - [13.839635°E, 50.564699°N]
Exporting all 128 bands
╭───────────────────┬──────────────────────────────────────────────────────────╮
│ Format:           │

### Measure the dequantization time

This measures only the conversion from the downloaded NPY files to georeferenced GeoTIFF tiles. VRT creation and mosaicking are deliberately left out.

The conversion function normally reads from `OUTPUT_ROOT` and writes to `TILE_TIFF_ROOT`. For the benchmark, both paths are temporarily redirected to the test folders and restored afterwards.

In [ ]:
# Temporarily point the conversion function to the benchmark folders.
_original_output_root = OUTPUT_ROOT
_original_tile_tiff_root = TILE_TIFF_ROOT

try:
    OUTPUT_ROOT = RUNTIME_DOWNLOAD_ROOT
    TILE_TIFF_ROOT = RUNTIME_TILE_TIFF_ROOT

    conversion_start = time.perf_counter()

    runtime_tile_tiffs = convert_year_tiles_to_tiffs(RUNTIME_YEAR)

    conversion_seconds = time.perf_counter() - conversion_start
    conversion_minutes = conversion_seconds / 60
    conversion_hours = conversion_seconds / 3600

    print()
    print("Converted GeoTIFF tiles:", len(runtime_tile_tiffs))
    print(
        f"TESSERA dequantization and GeoTIFF conversion for "
        f"{RUNTIME_YEAR} took {conversion_hours:.2f} hours "
        f"({conversion_minutes:.1f} minutes)."
    )

finally:
    # Restore the normal production paths, even if the benchmark fails.
    OUTPUT_ROOT = _original_output_root
    TILE_TIFF_ROOT = _original_tile_tiff_root

### Optional cleanup

The benchmark creates a second copy of one annual download and its GeoTIFF tiles. Remove the test folder afterwards when the runtime has been recorded.

In [ ]:
# Uncomment only after the benchmark results have been saved somewhere.
# import shutil
# shutil.rmtree(OUTPUT_ROOT / "_runtime_test")
# print("Runtime benchmark files deleted.")